# QuickPay FinTech Operations — Python Pipeline
**Part 3:** Reconciliation Workflow (ledger.csv vs gateway.csv)  
**Part 4:** JSON Normalization (api_response_sample.json)

All outputs saved to `01_data/processed/` and `04_python/`


---
# PART 3: Python Reconciliation Workflow

## 3.1 Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import json
import os

# Create output directories
os.makedirs('01_data/processed', exist_ok=True)
os.makedirs('04_python', exist_ok=True)

print("Libraries loaded successfully.")
print(f"Pandas version : {pd.__version__}")
print(f"NumPy  version : {np.__version__}")


## 3.2 Load Data

In [ ]:
ledger  = pd.read_csv('01_data/raw/ledger.csv')
gateway = pd.read_csv('01_data/raw/gateway.csv')

print(f"Ledger  : {len(ledger)}  rows  |  {ledger.shape[1]} columns")
print(f"Gateway : {len(gateway)} rows  |  {gateway.shape[1]} columns")
print()
print("=== LEDGER ===")
display(ledger)
print()
print("=== GATEWAY ===")
display(gateway)


## 3.3 Validation Checks
### 3.3.1 Data Types

In [ ]:
print("=== LEDGER DATA TYPES ===")
print(ledger.dtypes)
print()
print("=== GATEWAY DATA TYPES ===")
print(gateway.dtypes)


### 3.3.2 Null / Missing Values

In [ ]:
print("=== NULLS — LEDGER ===")
print(ledger.isnull().sum())
print()
print("=== NULLS — GATEWAY ===")
print(gateway.isnull().sum())

l_nulls = ledger.isnull().sum().sum()
g_nulls = gateway.isnull().sum().sum()

print(f"\nTotal nulls in Ledger  : {l_nulls}")
print(f"Total nulls in Gateway : {g_nulls}")
print("\n✅ No missing values found." if l_nulls == 0 and g_nulls == 0 else "\n⚠️ Missing values detected.")


### 3.3.3 Duplicate Check

In [ ]:
l_dupes = ledger.duplicated(subset='transaction_id').sum()
g_dupes = gateway.duplicated(subset='transaction_id').sum()

print(f"Duplicate transaction_ids in Ledger  : {l_dupes}")
print(f"Duplicate transaction_ids in Gateway : {g_dupes}")

if l_dupes == 0 and g_dupes == 0:
    print("\n✅ No duplicates found in either file.")
else:
    ledger  = ledger.drop_duplicates(subset='transaction_id')
    gateway = gateway.drop_duplicates(subset='transaction_id')
    print("\n⚠️ Duplicates removed.")


### 3.3.4 Basic Statistics

In [ ]:
print("=== LEDGER — AMOUNT STATS ===")
display(ledger[['amount_usd']].describe().round(2))
print()
print("=== GATEWAY — AMOUNT STATS ===")
display(gateway[['amount_usd']].describe().round(2))
print()
print("=== LEDGER STATUS COUNTS ===")
print(ledger['status'].value_counts())
print()
print("=== GATEWAY STATUS COUNTS ===")
print(gateway['status'].value_counts())


## 3.4 Reconciliation Logic
### 3.4.1 Missing in Gateway
Transactions in **Ledger** but NOT in **Gateway**.

In [ ]:
ledger_ids  = set(ledger['transaction_id'])
gateway_ids = set(gateway['transaction_id'])

missing_in_gateway = ledger[
    ledger['transaction_id'].isin(ledger_ids - gateway_ids)
].copy()
missing_in_gateway.reset_index(drop=True, inplace=True)

print(f"Records missing in Gateway: {len(missing_in_gateway)}")
display(missing_in_gateway)


### 3.4.2 Missing in Ledger
Transactions in **Gateway** but NOT in **Ledger**.

In [ ]:
missing_in_ledger = gateway[
    gateway['transaction_id'].isin(gateway_ids - ledger_ids)
].copy()
missing_in_ledger.reset_index(drop=True, inplace=True)

print(f"Records missing in Ledger: {len(missing_in_ledger)}")
display(missing_in_ledger)


### 3.4.3 Amount Mismatches
Transactions in **both** files but `amount_usd` values differ.

In [ ]:
common = pd.merge(
    ledger, gateway,
    on='transaction_id',
    suffixes=('_ledger', '_gateway')
)

amount_mismatches = common[
    common['amount_usd_ledger'] != common['amount_usd_gateway']
][['transaction_id',
   'transaction_date_ledger', 'merchant_id_ledger',
   'amount_usd_ledger', 'amount_usd_gateway',
   'status_ledger', 'payment_method_ledger']].copy()

amount_mismatches.columns = [
    'transaction_id', 'transaction_date', 'merchant_id',
    'amount_usd_ledger', 'amount_usd_gateway', 'status', 'payment_method'
]
amount_mismatches['amount_difference'] = (
    amount_mismatches['amount_usd_ledger'] - amount_mismatches['amount_usd_gateway']
).round(2)
amount_mismatches.reset_index(drop=True, inplace=True)

print(f"Amount mismatches found: {len(amount_mismatches)}")
display(amount_mismatches)


### 3.4.4 Status Mismatches
Transactions in **both** files but `status` values differ.

In [ ]:
status_mismatches = common[
    common['status_ledger'] != common['status_gateway']
][['transaction_id',
   'transaction_date_ledger', 'merchant_id_ledger',
   'amount_usd_ledger',
   'status_ledger', 'status_gateway',
   'payment_method_ledger']].copy()

status_mismatches.columns = [
    'transaction_id', 'transaction_date', 'merchant_id',
    'amount_usd', 'status_ledger', 'status_gateway', 'payment_method'
]
status_mismatches.reset_index(drop=True, inplace=True)

print(f"Status mismatches found: {len(status_mismatches)}")
display(status_mismatches)


## 3.5 Final Reconciliation Report
All issues combined into one master report.

In [ ]:
report_rows = []

# Missing in gateway
for _, r in missing_in_gateway.iterrows():
    report_rows.append({
        'transaction_id'    : r['transaction_id'],
        'transaction_date'  : r['transaction_date'],
        'merchant_id'       : r['merchant_id'],
        'amount_usd_ledger' : r['amount_usd'],
        'amount_usd_gateway': None,
        'status_ledger'     : r['status'],
        'status_gateway'    : None,
        'payment_method'    : r['payment_method'],
        'issue_type'        : 'missing_in_gateway'
    })

# Missing in ledger
for _, r in missing_in_ledger.iterrows():
    report_rows.append({
        'transaction_id'    : r['transaction_id'],
        'transaction_date'  : r['transaction_date'],
        'merchant_id'       : r['merchant_id'],
        'amount_usd_ledger' : None,
        'amount_usd_gateway': r['amount_usd'],
        'status_ledger'     : None,
        'status_gateway'    : r['status'],
        'payment_method'    : r['payment_method'],
        'issue_type'        : 'missing_in_ledger'
    })

# Amount mismatches
for _, r in amount_mismatches.iterrows():
    report_rows.append({
        'transaction_id'    : r['transaction_id'],
        'transaction_date'  : r['transaction_date'],
        'merchant_id'       : r['merchant_id'],
        'amount_usd_ledger' : r['amount_usd_ledger'],
        'amount_usd_gateway': r['amount_usd_gateway'],
        'status_ledger'     : r['status'],
        'status_gateway'    : r['status'],
        'payment_method'    : r['payment_method'],
        'issue_type'        : 'amount_mismatch'
    })

# Status mismatches
for _, r in status_mismatches.iterrows():
    report_rows.append({
        'transaction_id'    : r['transaction_id'],
        'transaction_date'  : r['transaction_date'],
        'merchant_id'       : r['merchant_id'],
        'amount_usd_ledger' : r['amount_usd'],
        'amount_usd_gateway': r['amount_usd'],
        'status_ledger'     : r['status_ledger'],
        'status_gateway'    : r['status_gateway'],
        'payment_method'    : r['payment_method'],
        'issue_type'        : 'status_mismatch'
    })

reconciliation_report = pd.DataFrame(report_rows)

print(f"Total reconciliation issues : {len(reconciliation_report)}")
print()
print("Issue breakdown:")
print(reconciliation_report['issue_type'].value_counts())
print()
display(reconciliation_report)


## 3.6 Summary Metrics
Generating `summary_metrics.json` with all required keys.

In [ ]:
ledger_total  = round(float(ledger['amount_usd'].sum()), 2)
gateway_total = round(float(gateway['amount_usd'].sum()), 2)

amount_at_risk = round(
    float(missing_in_gateway['amount_usd'].sum()) +
    float(missing_in_ledger['amount_usd'].sum()) +
    float(amount_mismatches['amount_difference'].abs().sum()),
    2
)

summary_metrics = {
    "total_ledger_rows"          : int(len(ledger)),
    "total_gateway_rows"         : int(len(gateway)),
    "missing_in_gateway_count"   : int(len(missing_in_gateway)),
    "missing_in_ledger_count"    : int(len(missing_in_ledger)),
    "amount_mismatch_count"      : int(len(amount_mismatches)),
    "status_mismatch_count"      : int(len(status_mismatches)),
    "reconciliation_issue_count" : int(len(reconciliation_report)),
    "ledger_total_amount"        : ledger_total,
    "gateway_total_amount"       : gateway_total,
    "amount_at_risk"             : amount_at_risk
}

print("=== SUMMARY METRICS ===")
for k, v in summary_metrics.items():
    print(f"  {k:<35}: {v}")


## 3.7 Save Part 3 Output Files

In [ ]:
# CSVs
missing_in_gateway.to_csv('01_data/processed/missing_in_gateway.csv', index=False)
missing_in_ledger.to_csv('01_data/processed/missing_in_ledger.csv', index=False)
amount_mismatches.to_csv('01_data/processed/amount_mismatches.csv', index=False)
status_mismatches.to_csv('01_data/processed/status_mismatches.csv', index=False)
reconciliation_report.to_csv('01_data/processed/reconciliation_report.csv', index=False)

# summary_metrics.json — goes into 04_python/
with open('04_python/summary_metrics.json', 'w') as f:
    json.dump(summary_metrics, f, indent=2)

print("✅ Part 3 files saved:")
print("   01_data/processed/missing_in_gateway.csv")
print("   01_data/processed/missing_in_ledger.csv")
print("   01_data/processed/amount_mismatches.csv")
print("   01_data/processed/status_mismatches.csv")
print("   01_data/processed/reconciliation_report.csv")
print("   04_python/summary_metrics.json")


---
# PART 4: JSON Normalization
Using `api_response_sample.json` — read → flatten → clean → save.

## 4.1 Load JSON File

In [ ]:
with open('01_data/raw/api_response_sample.json', 'r') as f:
    api_data = json.load(f)

print(f"Source       : {api_data['source']}")
print(f"Generated at : {api_data['generated_at']}")
print(f"Total batches: {len(api_data['batches'])}")
print()
# Preview raw structure
print("=== RAW JSON STRUCTURE PREVIEW ===")
for batch in api_data['batches']:
    print(f"  Batch: {batch['batch_id']} | Merchant: {batch['merchant']['merchant_name']} | Settlements: {len(batch['settlements'])}")


## 4.2 Flatten Nested JSON
The structure has 3 nesting levels:
`batches` → `merchant` + `settlements` → `bank`

Each settlement becomes one flat row.

In [ ]:
rows = []

for batch in api_data['batches']:
    batch_id      = batch['batch_id']
    merchant_id   = batch['merchant']['merchant_id']
    merchant_name = batch['merchant']['merchant_name']
    region        = batch['merchant']['region']

    for settlement in batch['settlements']:
        rows.append({
            'batch_id'      : batch_id,
            'merchant_id'   : merchant_id,
            'merchant_name' : merchant_name,
            'region'        : region,
            'settlement_id' : settlement['settlement_id'],
            'amount_usd'    : settlement['amount_usd'],
            'status'        : settlement['status'],
            'processed_at'  : settlement['processed_at'],
            'bank_name'     : settlement['bank']['name'],
            'bank_country'  : settlement['bank']['country'],
        })

api_df = pd.DataFrame(rows)

print(f"Total flattened rows : {len(api_df)}")
print(f"Columns              : {list(api_df.columns)}")
print()
display(api_df)


## 4.3 Clean Column Names & Convert Date/Time Fields

In [ ]:
# Clean column names — ensure snake_case, no spaces
api_df.columns = [col.strip().lower().replace(' ', '_') for col in api_df.columns]

# Convert processed_at to proper datetime
api_df['processed_at'] = pd.to_datetime(api_df['processed_at'])

# Split into separate date and time columns
api_df['processed_date'] = api_df['processed_at'].dt.strftime('%Y-%m-%d')
api_df['processed_time'] = api_df['processed_at'].dt.strftime('%H:%M:%S')

# Drop original datetime column (split version is cleaner)
api_df.drop(columns=['processed_at'], inplace=True)

# Standardize status to lowercase
api_df['status'] = api_df['status'].str.strip().str.lower()

# Reorder columns logically
api_df = api_df[[
    'batch_id', 'merchant_id', 'merchant_name', 'region',
    'settlement_id', 'amount_usd', 'status',
    'processed_date', 'processed_time',
    'bank_name', 'bank_country'
]]

print("=== CLEANED COLUMN NAMES ===")
print(api_df.columns.tolist())
print()
print("=== DATA TYPES ===")
print(api_df.dtypes)
print()
display(api_df)


## 4.4 Quick Analysis of Normalized Data

In [ ]:
print("=== STATUS BREAKDOWN ===")
print(api_df['status'].value_counts())
print()
print("=== AMOUNT BY STATUS ===")
print(api_df.groupby('status')['amount_usd'].sum().round(2))
print()
print("=== AMOUNT BY MERCHANT ===")
print(api_df.groupby('merchant_name')['amount_usd'].sum().round(2))
print()
total_settled = api_df[api_df['status'] == 'settled']['amount_usd'].sum()
print(f"Total Settled Amount : ${total_settled:,.2f}")


## 4.5 Save Normalized Output

In [ ]:
api_df.to_csv('01_data/processed/api_normalized.csv', index=False)

print("✅ Part 4 file saved:")
print("   01_data/processed/api_normalized.csv")
print(f"   Rows    : {len(api_df)}")
print(f"   Columns : {list(api_df.columns)}")


---
# Dashboard Source Files (Part 5 Support)
Generating processed files needed for Looker Studio dashboard.

In [ ]:
cleaned = pd.read_csv('01_data/processed/cleaned_transactions.csv')
print(f"Cleaned transactions loaded: {len(cleaned)} rows")
display(cleaned.head(3))


### Daily Summary

In [ ]:
daily_summary = cleaned.groupby('transaction_date').agg(
    total_gmv          = ('amount_usd', 'sum'),
    confirmed_gmv      = ('amount_usd', lambda x: x[cleaned.loc[x.index, 'status'] == 'captured'].sum()),
    total_transactions = ('transaction_id', 'count'),
    successful_count   = ('status', lambda x: (x == 'captured').sum()),
    failed_count       = ('status', lambda x: (x == 'failed').sum()),
    chargeback_count   = ('status', lambda x: (x == 'chargeback').sum()),
).reset_index()

daily_summary['success_rate_pct'] = (daily_summary['successful_count'] / daily_summary['total_transactions'] * 100).round(2)
daily_summary['total_gmv']        = daily_summary['total_gmv'].round(2)
daily_summary['confirmed_gmv']    = daily_summary['confirmed_gmv'].round(2)

daily_summary.to_csv('01_data/processed/daily_summary.csv', index=False)
print("✅ daily_summary.csv saved")
display(daily_summary)


### Payment Method Breakdown

In [ ]:
pm_breakdown = cleaned.groupby('payment_method').agg(
    total_transactions = ('transaction_id', 'count'),
    total_gmv          = ('amount_usd', 'sum'),
    confirmed_gmv      = ('amount_usd', lambda x: x[cleaned.loc[x.index, 'status'] == 'captured'].sum()),
    success_count      = ('status', lambda x: (x == 'captured').sum()),
    failed_count       = ('status', lambda x: (x == 'failed').sum()),
).reset_index()

pm_breakdown['success_rate_pct'] = (pm_breakdown['success_count'] / pm_breakdown['total_transactions'] * 100).round(2)
pm_breakdown['total_gmv']        = pm_breakdown['total_gmv'].round(2)
pm_breakdown['confirmed_gmv']    = pm_breakdown['confirmed_gmv'].round(2)

pm_breakdown.to_csv('01_data/processed/payment_method_breakdown.csv', index=False)
print("✅ payment_method_breakdown.csv saved")
display(pm_breakdown)


### Region Breakdown

In [ ]:
region_breakdown = cleaned.groupby('gateway_region').agg(
    total_transactions = ('transaction_id', 'count'),
    total_gmv          = ('amount_usd', 'sum'),
    confirmed_gmv      = ('amount_usd', lambda x: x[cleaned.loc[x.index, 'status'] == 'captured'].sum()),
    avg_risk_score     = ('risk_score', 'mean'),
    high_risk_count    = ('high_risk_flag', 'sum'),
    high_value_count   = ('high_value_flag', 'sum'),
).reset_index()

region_breakdown['avg_risk_score'] = region_breakdown['avg_risk_score'].round(2)
region_breakdown['total_gmv']      = region_breakdown['total_gmv'].round(2)
region_breakdown['confirmed_gmv']  = region_breakdown['confirmed_gmv'].round(2)

region_breakdown.to_csv('01_data/processed/region_breakdown.csv', index=False)
print("✅ region_breakdown.csv saved")
display(region_breakdown)


### Merchant Performance Summary

In [ ]:
merch_perf = cleaned.groupby(['merchant_id', 'merchant_name']).agg(
    total_transactions = ('transaction_id', 'count'),
    total_gmv          = ('amount_usd', 'sum'),
    confirmed_gmv      = ('amount_usd', lambda x: x[cleaned.loc[x.index, 'status'] == 'captured'].sum()),
    chargeback_count   = ('status', lambda x: (x == 'chargeback').sum()),
    failed_count       = ('status', lambda x: (x == 'failed').sum()),
    avg_risk_score     = ('risk_score', 'mean'),
    high_value_count   = ('high_value_flag', 'sum'),
    high_risk_count    = ('high_risk_flag', 'sum'),
).reset_index()

merch_perf['chargeback_ratio_pct'] = (merch_perf['chargeback_count'] / merch_perf['total_transactions'] * 100).round(2)
merch_perf['success_rate_pct']     = ((merch_perf['total_transactions'] - merch_perf['failed_count'] - merch_perf['chargeback_count']) / merch_perf['total_transactions'] * 100).round(2)
merch_perf['avg_risk_score']       = merch_perf['avg_risk_score'].round(2)
merch_perf['total_gmv']            = merch_perf['total_gmv'].round(2)
merch_perf['confirmed_gmv']        = merch_perf['confirmed_gmv'].round(2)

merch_perf.to_csv('01_data/processed/merchant_performance_summary.csv', index=False)
print("✅ merchant_performance_summary.csv saved")
display(merch_perf)


---
## ✅ All Done — File Checklist

### Part 3 — Reconciliation
| File | Location | Records |
|---|---|---|
| `missing_in_gateway.csv` | `01_data/processed/` | 2 |
| `missing_in_ledger.csv` | `01_data/processed/` | 1 |
| `amount_mismatches.csv` | `01_data/processed/` | 2 |
| `status_mismatches.csv` | `01_data/processed/` | 1 |
| `reconciliation_report.csv` | `01_data/processed/` | 6 |
| `summary_metrics.json` | `04_python/` | 10 keys |

### Part 4 — JSON Normalization
| File | Location | Records |
|---|---|---|
| `api_normalized.csv` | `01_data/processed/` | 6 |

### Dashboard Sources
| File | Location |
|---|---|
| `daily_summary.csv` | `01_data/processed/` |
| `payment_method_breakdown.csv` | `01_data/processed/` |
| `region_breakdown.csv` | `01_data/processed/` |
| `merchant_performance_summary.csv` | `01_data/processed/` |
